# Paper 3: PCA + ANN for transformer fault classification
Nkwambe & Thango, Energies 2026, 19, 2947.
Upload DuVal.csv to Colab first (left sidebar > Files > Upload).

In [ ]:
# Cell 1 - imports, load data, REMOVE DUPLICATES
import numpy as np, pandas as pd, warnings
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
warnings.filterwarnings("ignore")

df = pd.read_csv("DuVal.csv")
df["Fault"] = df["Fault"].str.strip()
print("Duplicate rows removed:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print(df.shape); print(df["Fault"].value_counts())
y = df["Fault"].values; classes = sorted(set(y))

In [ ]:
# Cell 2 - feature engineering: 15 features (5 gases + TCH + 9 ratios), eps = 1e-6
EPS = 1e-6
g = df[["H2","CH4","C2H6","C2H4","C2H2"]].astype(float).copy()
g["TCH"] = g["CH4"] + g["C2H6"] + g["C2H4"] + g["C2H2"]                  # Eq. (1)
F = g.copy()
F["CH4/H2"]    = g["CH4"]  / (g["H2"]   + EPS)
F["C2H2/C2H4"] = g["C2H2"] / (g["C2H4"] + EPS)
F["C2H4/C2H6"] = g["C2H4"] / (g["C2H6"] + EPS)
F["C2H2/CH4"]  = g["C2H2"] / (g["CH4"]  + EPS)
F["C2H6/CH4"]  = g["C2H6"] / (g["CH4"]  + EPS)
F["C2H4/TCH"]  = g["C2H4"] / (g["TCH"]  + EPS)
F["C2H6/TCH"]  = g["C2H6"] / (g["TCH"]  + EPS)
F["CH4/TCH"]   = g["CH4"]  / (g["TCH"]  + EPS)
F["C2H2/TCH"]  = g["C2H2"] / (g["TCH"]  + EPS)
print("Feature matrix:", F.shape)

# feature sets for the comparison in Table 4
RAW5 = F[["H2","CH4","C2H6","C2H4","C2H2"]]
IEC3 = F[["C2H2/C2H4","CH4/H2","C2H4/C2H6"]]          # IEC three-ratio method
ALL15 = F

In [ ]:
# Cell 3 - correlation matrix (paper Fig. 3)
corr = F.corr()
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(15)); ax.set_xticklabels(F.columns, rotation=90); ax.set_yticks(range(15)); ax.set_yticklabels(F.columns)
for i in range(15):
    for j in range(15): ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=6)
plt.colorbar(im); plt.title("Pearson correlation of the 15 features"); plt.tight_layout()
plt.savefig("fig_correlation.png", dpi=120); plt.show()

In [ ]:
# Cell 4 - stratified 80/20 split, standardise + PCA (fitted on TRAINING data only)
Xtr, Xte, ytr, yte = train_test_split(ALL15.values, y, test_size=0.2, random_state=42, stratify=y)
print("train/test:", Xtr.shape[0], Xte.shape[0])
scaler = StandardScaler().fit(Xtr)                      # Eq. (2)
pca = PCA().fit(scaler.transform(Xtr))                  # Eq. (3)-(4) via SVD
cum = np.cumsum(pca.explained_variance_ratio_) * 100
print("Cumulative variance (%):", np.round(cum, 1))
print("k for 95%:", int(np.argmax(cum >= 95)) + 1, "| k for 99%:", int(np.argmax(cum >= 99)) + 1)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].bar(range(1, 16), pca.explained_variance_ratio_*100, label="Individual")
ax[0].plot(range(1, 16), cum, "r-o", label="Cumulative"); ax[0].axhline(95, ls="--", c="gray"); ax[0].axhline(99, ls=":", c="orange")
ax[0].set_xlabel("Principal component"); ax[0].set_ylabel("Explained variance (%)"); ax[0].set_title("Scree plot"); ax[0].legend()
load = pd.DataFrame(pca.components_[:3].T, index=F.columns, columns=["PC1","PC2","PC3"])
load.plot.barh(ax=ax[1]); ax[1].set_title("Loadings of the first 3 PCs"); plt.tight_layout()
plt.savefig("fig_pca.png", dpi=120); plt.show()
print(load.round(2))

In [ ]:
# Cell 5 - ANN definition (Eq. 5-6): 1 hidden layer, 20 tanh neurons, softmax output, L-BFGS, L2 alpha=0.01
def ann(seed=42):
    return MLPClassifier(hidden_layer_sizes=(20,), activation="tanh", solver="lbfgs",
                         alpha=0.01, max_iter=2000, random_state=seed)

In [ ]:
# Cell 6 - sweep k = 1..15 (train acc, test acc, and 5-fold CV acc on the training set)
skf = StratifiedKFold(5, shuffle=True, random_state=42)
rows = []
for k in range(1, 16):
    pipe = make_pipeline(StandardScaler(), PCA(n_components=k), ann())
    pipe.fit(Xtr, ytr)
    cv = cross_val_score(make_pipeline(StandardScaler(), PCA(n_components=k), ann()), Xtr, ytr, cv=skf).mean()
    rows.append({"k": k, "CumVar%": cum[k-1], "Train%": accuracy_score(ytr, pipe.predict(Xtr))*100,
                 "Test%": accuracy_score(yte, pipe.predict(Xte))*100, "CV5%": cv*100})
sweep = pd.DataFrame(rows).round(1); print(sweep.to_string(index=False))
k_paper = int(sweep.loc[sweep["Test%"].idxmax(), "k"])      # paper's rule: best TEST accuracy (optimistic)
k_cv    = int(sweep.loc[sweep["CV5%"].idxmax(), "k"])       # fairer rule: best CV accuracy on training data
print(f"\nk chosen by best test accuracy (paper's rule): {k_paper}")
print(f"k chosen by best 5-fold CV on training data  : {k_cv}   <- unbiased choice")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(sweep.k, sweep["Train%"], "b-s", label="Train"); ax.plot(sweep.k, sweep["Test%"], "r-o", label="Test")
ax.plot(sweep.k, sweep["CV5%"], "g-^", label="5-fold CV (train)"); ax.axvline(k_cv, ls="--", c="green")
ax.set_xlabel("Number of principal components k"); ax.set_ylabel("Accuracy (%)"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig("fig_k_sweep.png", dpi=120); plt.show()

In [ ]:
# Cell 7 - final PCA+ANN model: confusion matrix and per-class metrics (k chosen by CV)
K = k_cv
final = make_pipeline(StandardScaler(), PCA(n_components=K), ann()).fit(Xtr, ytr)
yp = final.predict(Xte)
print(f"PCA + ANN (k={K}) test accuracy: {accuracy_score(yte, yp)*100:.1f}%")
print(classification_report(yte, yp, digits=3, zero_division=0))
ConfusionMatrixDisplay(confusion_matrix(yte, yp, labels=classes), display_labels=classes).plot(cmap="Blues")
plt.title(f"Confusion matrix, PCA+ANN (k={K})"); plt.savefig("fig_confusion.png", dpi=120); plt.show()

In [ ]:
# Cell 8 - comparisons: feature sets (Table 4) and classifiers (Table 5), averaged over 10 random splits
N_SEEDS = 10
def models(n_feat):
    return {
     "ANN, raw 5 gases":      (RAW5,  lambda s: make_pipeline(StandardScaler(), ann(s))),
     "ANN, IEC 3 ratios":     (IEC3,  lambda s: make_pipeline(StandardScaler(), ann(s))),
     "ANN, all 15 (no PCA)":  (ALL15, lambda s: make_pipeline(StandardScaler(), ann(s))),
     f"PCA + ANN (k={K})":    (ALL15, lambda s: make_pipeline(StandardScaler(), PCA(n_components=K), ann(s))),
     "GNB (15)":              (ALL15, lambda s: make_pipeline(StandardScaler(), GaussianNB())),
     "KNN k=3 (15)":          (ALL15, lambda s: make_pipeline(StandardScaler(), KNeighborsClassifier(3))),
     "KNN k=5 (15)":          (ALL15, lambda s: make_pipeline(StandardScaler(), KNeighborsClassifier(5))),
     "SVM RBF (15)":          (ALL15, lambda s: make_pipeline(StandardScaler(), SVC(kernel="rbf"))),
    }
res = []
for seed in range(N_SEEDS):
    a, b, ya, yb = train_test_split(np.arange(len(df)), y, test_size=0.2, random_state=seed, stratify=y)
    for name, (X, mk) in models(15).items():
        m = mk(seed).fit(X.values[a], ya)
        res.append({"Model": name, "Seed": seed,
                    "Train": accuracy_score(ya, m.predict(X.values[a]))*100,
                    "Test": accuracy_score(yb, m.predict(X.values[b]))*100})
res = pd.DataFrame(res)
table = res.groupby("Model")[["Train","Test"]].agg(["mean","std"]).round(1)
print(f"\n=== Mean over {N_SEEDS} random stratified 80/20 splits (accuracy %) ===")
print(table.sort_values(("Test","mean"), ascending=False))
m = res.groupby("Model")["Test"].agg(["mean","std"]).sort_values("mean")
plt.figure(figsize=(8, 5)); plt.barh(m.index, m["mean"], xerr=m["std"]); plt.xlabel("Test accuracy (%)")
plt.title("Feature sets and classifiers"); plt.tight_layout(); plt.savefig("fig_comparison.png", dpi=120); plt.show()
sweep.to_csv("k_sweep.csv", index=False); res.to_csv("comparison_runs.csv", index=False)